# Jacobi Identity Verification for IMSRG Commutators

This notebook uses `qcombo` to **verify the Jacobi identity** for nested commutators appearing in the IMSRG formalism.

## Theoretical Background

The **Jacobi identity** states that for any three operators $X$, $Y$, $Z$:

$$[X, [Y, Z]] + [Y, [Z, X]] + [Z, [X, Y]] = 0$$

In the IMSRG framework, operators are decomposed by body rank:

$$X = X_{1B} + X_{2B} + \cdots$$

The Jacobi identity must hold for each **body-rank channel** of the nested commutators. For a given outer contraction body $K$ and inner contraction body $L$:

$$[X_{m_1}, [Y_{m_2}, Z_{m_3}]_L]_K + [Y_{m_1}, [Z_{m_2}, X_{m_3}]_L]_K + [Z_{m_1}, [X_{m_2}, Y_{m_3}]_L]_K \stackrel{?}{=} 0$$

Verification involves:
1. Computing the inner commutator $[Y, Z]_L$
2. Substituting it into the outer commutator $[X, \cdot]_K$
3. Summing cyclic permutations and checking if the result vanishes

**Reference**: The Jacobi identity is a fundamental consistency check for any Lie algebraic structure. In IMSRG, it ensures that the flow equations are well-defined.

---
## 1. Setup: Imports and Helper Functions

In [1]:
import qcombo
from qcombo.tools import Filter, indicesMultToSimp
from qcombo.canonical import canonicalize
from qcombo import simplifyUseBoth, simplifyUseDummyIndices
from sympy import IndexedBase, symbols, sympify
from sympy import preorder_traversal, IndexedBase, factorial, Rational
from sympy.combinatorics import Permutation
from itertools import permutations
from sympy import simplify, expand
from sympy import symbols, preorder_traversal
from sympy.tensor.indexed import Indexed
from sympy.core.mul import Mul
from sympy.core.add import Add
from sympy import S
from IPython.display import display, Latex
import time

# Define tensor symbols
A = IndexedBase('A')
G = IndexedBase('G')
H = IndexedBase('H')
R = IndexedBase('R')
delta = IndexedBase(chr(948))

# Additional index symbols
t = IndexedBase('t')
f = IndexedBase('f')
p = IndexedBase('p')
q = IndexedBase('q')
s = IndexedBase('s')
r = IndexedBase('r')
u = IndexedBase('u')
v = IndexedBase('v')

print(f"qcombo version: {qcombo.__version__}")

qcombo version: 0.2.0


### 1.1 Tensor Utility Functions

In [2]:
def find_tensor(expr, tensor="A"):
    """Find the first occurrence of a tensor in an expression."""
    for term in preorder_traversal(expr):
        if isinstance(term, Indexed) and term.base == IndexedBase(tensor):
            return term
    return None


def find_A_tensor(expr):
    """Find the A tensor (contraction tensor) in an expression."""
    for term in preorder_traversal(expr):
        if isinstance(term, Indexed) and term.base == A:
            return term
    return None


def replace_A_tensor(expr, replace_term=1):
    """Replace the A tensor in an expression with a scalar."""
    replace_dict = {}
    for term in preorder_traversal(expr):
        if isinstance(term, Indexed) and term.base == A:
            replace_dict[term] = replace_term
    return expr.xreplace(replace_dict)


def get_all_indices(expr):
    """Get all indices appearing in an expression."""
    indices_set = set()
    for term in preorder_traversal(expr):
        if isinstance(term, Indexed):
            for i in term.indices[0]:  # upper indices
                indices_set.add(i)
            for i in term.indices[1]:  # lower indices
                indices_set.add(i)
    return sorted(indices_set, key=str)


def getIndicesFromExpr(expr):
    """
    Extract sum indices and contraction indices from expression.
    
    Returns:
        (sum_indices, contraction_indices): sum indices are dummy/summed indices,
        contraction_indices are the free indices carried by the A tensor
    """
    all_indices = get_all_indices(expr)
    
    A_tensor = find_A_tensor(expr)
    
    if A_tensor is None:  # Expression contracts to zero-body term
        contraction_indices = [(), ()]
        sum_indices = all_indices
    else:
        contraction_indices = A_tensor.indices
        sum_indices = set(all_indices) - set(get_all_indices(A_tensor))
    
    sum_indices = sorted(sum_indices, key=str)
    return sum_indices, contraction_indices


print("Tensor utility functions defined.")

Tensor utility functions defined.


### 1.2 Index Permutation and Anti-symmetrization

These functions handle index swapping and anti-symmetrization, which are essential for restoring proper fermionic antisymmetry in the commutator expressions.

In [3]:
def swap_indices(expr, idx_a, idx_b):
    """
    Swap two indices everywhere in the expression.
    Equivalent to applying permutation operator P_{ab}.
    
    Parameters:
        expr: SymPy expression
        idx_a, idx_b: indices to swap (SymPy symbols)
    
    Returns:
        Expression with idx_a and idx_b swapped
    
    Example:
        swap_indices(A[a,b]*B[c,a], a, b)  -->  A[b,a]*B[c,b]
    """
    if idx_a == idx_b:
        return expr
    
    if type(idx_a) == str:
        idx_a = IndexedBase(idx_a)
    if type(idx_b) == str:
        idx_b = IndexedBase(idx_b)
    
    # Use a temporary symbol to avoid conflicts during swap
    tmp = symbols('__tmp_swap__')
    return expr.xreplace({idx_a: tmp, idx_b: idx_a}).xreplace({tmp: idx_b})


def _perm_sign(original, permuted):
    """
    Calculate the sign of a permutation (+1 for even, -1 for odd).
    """
    n = len(original)
    if n <= 1:
        return 1
    
    mapping = {v: i for i, v in enumerate(original)}
    perm_list = [mapping[v] for v in permuted]
    
    inversions = 0
    for i in range(n):
        for j in range(i + 1, n):
            if perm_list[i] > perm_list[j]:
                inversions += 1
    
    return 1 if inversions % 2 == 0 else -1


def antisymmetrize(expr, up_indices, lo_indices):
    """
    Anti-symmetrize the expression with respect to specified upper and lower index groups.
    
    Applies the operator: 1/(n! * m!) * prod_{pairs} (1 - P_{pair})
    
    For up_indices = [a,b] and lo_indices = [c,d]:
        result = 1/4 * (1-P_{ab})(1-P_{cd}) * expr
               = 1/4 * (expr - P_{ab}*expr - P_{cd}*expr + P_{ab}*P_{cd}*expr)
    
    This ensures: result^{ab}_{cd} = -result^{ba}_{cd} = -result^{ab}_{dc} = result^{ba}_{dc}
    
    Parameters:
        expr: SymPy expression
        up_indices: list of upper indices to anti-symmetrize, e.g., [a, b]
        lo_indices: list of lower indices to anti-symmetrize, e.g., [c, d]
    
    Returns:
        Anti-symmetrized expression (normalized by 1/(n! * m!))
    """
    result = 0
    
    for up_perm in permutations(up_indices):
        for lo_perm in permutations(lo_indices):
            replace_dict = {}
            for old, new in zip(up_indices, up_perm):
                if old != new:
                    replace_dict[old] = new
            for old, new in zip(lo_indices, lo_perm):
                if old != new:
                    replace_dict[old] = new
            
            up_sign = _perm_sign(up_indices, list(up_perm))
            lo_sign = _perm_sign(lo_indices, list(lo_perm))
            sign = up_sign * lo_sign
            
            if replace_dict:
                term = expr.xreplace(replace_dict)
            else:
                term = expr
            
            result += sign * term
    
    n = len(up_indices)
    m = len(lo_indices)
    normalization = S(1) / (factorial(n) * factorial(m))
    
    return expand(result * normalization)


def antisymmetrize_expr(expr):
    """
    Anti-symmetrize the expression with respect to the A tensor's indices.
    Extracts indices from A, anti-symmetrizes the coefficient part, then re-attaches A.
    """
    A_tensor = find_A_tensor(expr)
    if A_tensor is not None:
        up_indices, lo_indices = A_tensor.indices[0], A_tensor.indices[1]
        tem_expr = expr / A_tensor
        antisymmetrized_expr = antisymmetrize(tem_expr, up_indices, lo_indices)
        return antisymmetrized_expr * A_tensor
    else:
        return expr


def antisymmetrize_tensor(expr, tensor):
    """Anti-symmetrize the expression with respect to a specific tensor's indices."""
    To_tensor = find_tensor(expr, tensor)
    if To_tensor is not None:
        up_indices, lo_indices = To_tensor.indices[0], To_tensor.indices[1]
        return antisymmetrize(expr, up_indices, lo_indices)
    else:
        return expr


print("Anti-symmetrization functions defined.")

Anti-symmetrization functions defined.


### 1.3 Display Helper

In [4]:
def jupyterDisplay(expr, title=None):
    """
    Display SymPy expression in LaTeX format in Jupyter Notebook.
    """
    if expr == 0 or expr is None:
        display(Latex(r"$$0$$"))
        return
    latex_expr = qcombo.texExp(expr)
    if title:
        print(title)
    display(Latex(f"$${latex_expr}$$"))


print("Display helper defined.")

Display helper defined.


---
## 2. Nested Commutator Computation

The core function `nested_commutator` computes expressions of the form:

$$[X_{m_1}, [Y_{m_2}, Z_{m_3}]_L]_K$$

where:
- $m_1, m_2, m_3$ are the body ranks of $X, Y, Z$
- $L$ is the contraction body of the inner commutator
- $K$ is the contraction body of the outer commutator

**Key steps:**
1. Compute the inner commutator $[Y, Z]_L$ via `easyCombo`
2. Extract its tensor structure (indices, coefficients)
3. Compute the outer commutator $[X, \cdot]_K$ via `easyCombo`
4. Substitute the inner result into the outer commutator
5. Apply anti-symmetrization and canonical simplification

In [5]:
def nested_commutator(outer_body=[1, 1, 0], inner_body=[1, 1, 1],
                      Base=["R", "G", "H"], commutator_dict=None, **kwargs):
    """
    Compute a nested commutator: [X_{m1}, [Y_{m2}, Z_{m3}]_L]_K
    
    Parameters:
        outer_body: [m1, L, K]  -- outer commutator: [X_{m1}, (inner result)_L]_K
        inner_body: [m2, m3, L] -- inner commutator: [Y_{m2}, Z_{m3}]_L
        Base: [X_base, Y_base, Z_base] -- base names for X, Y, Z
        commutator_dict: pre-computed dictionary of commutator expressions
    
    Returns:
        Canonicalized expression for the nested commutator
    """
    parallel = kwargs.get('parallel', False)
    show_process = kwargs.get('show_process', False)
    
    if max(outer_body) > 2 or max(inner_body) > 2:
        parallel = True
        show_process = True
    
    # Unpack body ranks
    outer_left_body = outer_body[0]
    outer_right_body = outer_body[1]
    outer_contraction_body = outer_body[2]
    
    inner_left_body = inner_body[0]
    inner_right_body = inner_body[1]
    inner_contraction_body = inner_body[2]
    
    # A nested commutator with 0B part is trivially zero
    if (outer_left_body == 0 or outer_right_body == 0) or (0 in inner_body):
        return 0
    
    # Consistency check: inner contraction body must equal outer right body
    if inner_contraction_body != outer_right_body:
        return 0
    
    # Check if contraction is possible: [m,n]_k requires m+n > k
    if (outer_left_body + outer_right_body <= outer_contraction_body) or \
       (inner_left_body + inner_right_body <= inner_contraction_body):
        return 0
    
    # --- Get outer and inner commutator expressions ---
    outer_commutator_expr = 0
    inner_commutator_expr = 0
    
    if commutator_dict is not None:
        try:
            outer_body_key = "commutator_" + "".join(map(str, outer_body))
            inner_body_key = "commutator_" + "".join(map(str, inner_body))
            outer_commutator_expr = commutator_dict[outer_body_key]
            inner_commutator_expr = commutator_dict[inner_body_key]
        except KeyError:
            print(f"Commutator not in dict for {outer_body}, {inner_body}. Computing...")
            outer_expr = qcombo.easyCombo(outer_left_body, outer_right_body,
                                          outer_contraction_body,
                                          show_process=show_process,
                                          savefile=False, parallel=parallel)
            inner_expr = qcombo.easyCombo(inner_left_body, inner_right_body,
                                          inner_contraction_body,
                                          show_process=show_process,
                                          savefile=False, parallel=parallel)
            for key, expr in outer_expr.expr_dict.items():
                outer_commutator_expr += expr
            for key, expr in inner_expr.expr_dict.items():
                inner_commutator_expr += expr
    else:
        outer_expr = qcombo.easyCombo(outer_left_body, outer_right_body,
                                      outer_contraction_body,
                                      show_process=show_process,
                                      savefile=False, parallel=parallel)
        inner_expr = qcombo.easyCombo(inner_left_body, inner_right_body,
                                      inner_contraction_body,
                                      show_process=show_process,
                                      savefile=False, parallel=parallel)
        for key, expr in outer_expr.expr_dict.items():
            outer_commutator_expr += expr
        for key, expr in inner_expr.expr_dict.items():
            inner_commutator_expr += expr
    
    # --- Extract index information ---
    outer_all_indices = get_all_indices(outer_commutator_expr)
    inner_all_indices = get_all_indices(inner_commutator_expr)
    
    outer_sum_idx, outer_con_idx = getIndicesFromExpr(outer_commutator_expr)
    inner_sum_idx, inner_con_idx = getIndicesFromExpr(inner_commutator_expr)
    
    # --- Apply coefficient normalization: (K!)^2 / (M! * N!)^2 ---
    outer_coef = factorial(outer_contraction_body)**2 / \
                 (factorial(outer_left_body) * factorial(outer_right_body))**2
    inner_coef = factorial(inner_contraction_body)**2 / \
                 (factorial(inner_left_body) * factorial(inner_right_body))**2
    
    outer_commutator_expr = outer_commutator_expr * outer_coef
    inner_commutator_expr = inner_commutator_expr * inner_coef
    
    # Restore antisymmetry
    outer_commutator_expr = antisymmetrize_expr(outer_commutator_expr)
    inner_commutator_expr = antisymmetrize_expr(inner_commutator_expr)
    
    # Factor out the A tensor from inner expression
    inner_commutator_expr = inner_commutator_expr / A[inner_con_idx]
    
    # --- Build the substitution function for outer's H tensor ---
    def replace_outer_Right_ME(indices):
        """Replace H tensor in outer commutator with the inner commutator result."""
        replace_dict = {}
        new_up_idx, new_lo_idx = indices[0], indices[1]
        old_up_idx, old_lo_idx = inner_con_idx[0], inner_con_idx[1]
        
        # Generate fresh dummy indices for inner sum indices
        indiceListPre = ['z', 'y', 'x', 'w', 'v', 'u', 't', 's', 'r', 'q', 'p',
                         'o', 'n', 'm', 'l', 'k', 'j', 'i', 'h', 'g', 'f', 'e',
                         'd', 'c', 'b', 'a']
        indiceListPre = [IndexedBase(i) for i in indiceListPre]
        pre_sum_idx = [i for i in indiceListPre
                       if i not in outer_all_indices and i not in inner_all_indices]
        
        for i in range(len(inner_sum_idx)):
            replace_dict[inner_sum_idx[i]] = pre_sum_idx[i]
        
        for i in range(len(new_up_idx)):
            replace_dict[old_up_idx[i]] = new_up_idx[i]
            replace_dict[old_lo_idx[i]] = new_lo_idx[i]
        
        return inner_commutator_expr.xreplace(replace_dict)
    
    # --- Substitute: replace G → R (to mark the outer left operator) ---
    outer_commutator_expr = outer_commutator_expr.replace(G, R)
    
    # Replace each H tensor in the outer commutator with the inner result
    replace_dict = {}
    for term in preorder_traversal(outer_commutator_expr):
        if isinstance(term, Indexed) and term.base == H:
            indices = term.indices
            replace_dict[term] = replace_outer_Right_ME(indices)
    
    nested_commutator_expr = outer_commutator_expr.xreplace(replace_dict)
    
    # --- Rename bases: R → Base[0], G → Base[1], H → Base[2] ---
    replaceBase_dict = {}
    replaceBase_dict[R] = IndexedBase(Base[0])
    replaceBase_dict[G] = IndexedBase(Base[1])
    replaceBase_dict[H] = IndexedBase(Base[2])
    nested_commutator_expr = nested_commutator_expr.xreplace(replaceBase_dict)
    
    # --- Canonical simplification ---
    canonTemp = canonicalize(nested_commutator_expr.expand(),
                             parallel=parallel, show_process=False)
    canon, indicesSet = indicesMultToSimp(canonTemp,
                                           parallel=parallel, show_process=False)
    
    simplified_expr = simplifyUseBoth(canon, parallel=parallel, show_process=False)
    full_simplified_expr = simplifyUseDummyIndices(simplified_expr,
                                                    parallel=parallel, show_process=False)
    
    return canon


print("nested_commutator function defined.")

nested_commutator function defined.


---
## 3. Warm-up: Single Nested Commutator

Before testing the full Jacobi identity, let's verify that `nested_commutator` gives the correct result for a simple case.

The simplest non-trivial nested commutator is $[X_1, [Y_1, Z_1]_1]_0$ (all 1-body operators).

In [6]:
# Test: [X_1B, [Y_1B, Z_1B]_1B]_0B
print("=" * 60)
print("Computing [X_1, [Y_1, Z_1]_1]_0 ...")
print("=" * 60)

t0 = time.time()
expr = nested_commutator([1, 1, 0], [1, 1, 1], Base=['X', 'Y', 'Z'])
t1 = time.time()

print(f"Computation time: {t1 - t0:.2f}s")
print()
print("Result: [X_1, [Y_1, Z_1]_1]_0 =")
jupyterDisplay(expr)
print()
print("This should match the known analytic result:")
print(r"  [X_1, [Y_1, Z_1]_1]_0 = \sum_{abc} (n_a - n_b) X^a_b (Y^b_c Z^c_a - Y^c_a Z^b_c)")

Computing [X_1, [Y_1, Z_1]_1]_0 ...
Computation time: 0.22s

Result: [X_1, [Y_1, Z_1]_1]_0 =


<IPython.core.display.Latex object>


This should match the known analytic result:
  [X_1, [Y_1, Z_1]_1]_0 = \sum_{abc} (n_a - n_b) X^a_b (Y^b_c Z^c_a - Y^c_a Z^b_c)


---
## 4. Jacobi Identity: Single Channel

The function `jacobi_identity_OneNested` tests the Jacobi identity for a **single body-rank channel**:

$$[X_{m_1}, [Y_{m_2}, Z_{m_3}]_L]_K + [Y_{m_1}, [Z_{m_2}, X_{m_3}]_L]_K + [Z_{m_1}, [X_{m_2}, Y_{m_3}]_L]_K = 0$$

This checks one specific combination of body ranks.

In [7]:
def jacobi_identity_OneNested(outer_body=[1, 1, 0], inner_body=[1, 1, 1],
                               Base=['X', 'Y', 'Z'], commutator_dict=None):
    """
    Test Jacobi identity for a single body-rank channel.
    
    Verifies:
        [X_{m1}, [Y_{m2}, Z_{m3}]_L]_K
      + [Y_{m1}, [Z_{m2}, X_{m3}]_L]_K
      + [Z_{m1}, [X_{m2}, Y_{m3}]_L]_K  ==  0
    
    Parameters:
        outer_body: [m1, L, K]
        inner_body: [m2, m3, L]
        Base: base names for [X, Y, Z]
        commutator_dict: pre-computed commutator dictionary
    
    Returns:
        Sum of the three cyclic terms (should be 0 if Jacobi holds)
    """
    X,Y,Z = Base[0],Base[1],Base[2]
    out_left, out_right, out_con = outer_body[0],outer_body[1],outer_body[2]
    in_left,in_right = inner_body[0],inner_body[1]

    Base1 = [Base[0], Base[1], Base[2]]
    expr_1 = nested_commutator(outer_body, inner_body, Base1, commutator_dict)
    str_1 = f"[{X}_{out_left},[{Y}_{in_left},{Z}_{in_right}]_{out_right}]_{out_con} "
    
    Base2 = [Base[1], Base[2], Base[0]]
    expr_2 = nested_commutator(outer_body, inner_body, Base2, commutator_dict)
    str_2 = f"[{Y}_{out_left},[{Z}_{in_left},{X}_{in_right}]_{out_right}]_{out_con}"

    
    Base3 = [Base[2], Base[0], Base[1]]
    expr_3 = nested_commutator(outer_body, inner_body, Base3, commutator_dict)
    str_3 = f"[{Z}_{out_left},[{X}_{in_left},{Y}_{in_right}]_{out_right}]_{out_con} "

    
    total = expr_1 + expr_2 + expr_3

    print(str_1+"=")
    jupyterDisplay(expr_1)
    print(str_2+"=")
    jupyterDisplay(expr_2)
    print(str_3+"=")
    jupyterDisplay(expr_3)

    print(str_1 +"+"+ str_2 +"+"+  str_3 + "=", total)

    
    if total == 0:
        print(f"[PASS] Jacobi identity holds for [{outer_body}, {inner_body}].")
    else:
        print(f"[FAIL] Jacobi identity FAILS for [{outer_body}, {inner_body}]!")
        print(f"Sum =")
        jupyterDisplay(total)
    
    return total


print("jacobi_identity_OneNested function defined.")

jacobi_identity_OneNested function defined.


In [8]:
# Test: Jacobi identity for [X_1, [Y_1, Z_1]_1]_0
print("=" * 60)
print("Testing Jacobi identity: [1,[1,1]_1]_0")
print("=" * 60)

t0 = time.time()
result = jacobi_identity_OneNested(
    outer_body=[1, 1, 0],
    inner_body=[1, 1, 1],
    Base=['X', 'Y', 'Z']
)
t1 = time.time()

print(f"Computation time: {t1 - t0:.2f}s")
print(f"\nExpected: 0")
print(f"Got: {result}")

Testing Jacobi identity: [1,[1,1]_1]_0
[X_1,[Y_1,Z_1]_1]_0 =


<IPython.core.display.Latex object>

[Y_1,[Z_1,X_1]_1]_0=


<IPython.core.display.Latex object>

[Z_1,[X_1,Y_1]_1]_0 =


<IPython.core.display.Latex object>

[X_1,[Y_1,Z_1]_1]_0 +[Y_1,[Z_1,X_1]_1]_0+[Z_1,[X_1,Y_1]_1]_0 = 0
[PASS] Jacobi identity holds for [[1, 1, 0], [1, 1, 1]].
Computation time: 0.15s

Expected: 0
Got: 0


---
## 5. Run: 1-Body Operators Only

First, test the Jacobi identity for operators containing **only 1-body terms**:

$$X = X_{1B}, \quad Y = Y_{1B}, \quad Z = Z_{1B}$$

The only non-trivial nested commutator is $[1,[1,1]_1]_0 and [1,[1,1]_1]_1$.

In [9]:
print("=" * 60)
print("Jacobi Identity Test: 1-Body Operators (BaseBody = [1])")
print("=" * 60)

#
t_start = time.time()
result_110_111 = jacobi_identity_OneNested(
                outer_body=[1,1,0],inner_body=[1,1,1],
                Base=['X', 'Y', 'Z'])
t_end = time.time()

print(f"\nTotal computation time: {t_end - t_start:.2f}s")

t_start = time.time()
result_110_111 = jacobi_identity_OneNested(
    outer_body=[1,1,1],inner_body=[1,1,1],
    Base=['X', 'Y', 'Z']
)
t_end = time.time()

print(f"\nTotal computation time: {t_end - t_start:.2f}s")



Jacobi Identity Test: 1-Body Operators (BaseBody = [1])
[X_1,[Y_1,Z_1]_1]_0 =


<IPython.core.display.Latex object>

[Y_1,[Z_1,X_1]_1]_0=


<IPython.core.display.Latex object>

[Z_1,[X_1,Y_1]_1]_0 =


...

[Y_1,[Z_1,X_1]_1]_1=


<IPython.core.display.Latex object>

[Z_1,[X_1,Y_1]_1]_1 =


<IPython.core.display.Latex object>

[X_1,[Y_1,Z_1]_1]_1 +[Y_1,[Z_1,X_1]_1]_1+[Z_1,[X_1,Y_1]_1]_1 = 0
[PASS] Jacobi identity holds for [[1, 1, 1], [1, 1, 1]].

Total computation time: 0.18s
